# Tile Boundary Grid Generation for Web Mercator (WGS84)

This notebook provides a workflow to generate a grid of tile boundaries at a specific zoom level based on an Area of Interest (AOI). The output can be either a GeoJSON or a CSV file (with WKT geometries).

## 1. Install Necessary Libraries

We will use the following libraries:
- [mercantile](https://github.com/mapbox/mercantile): For Spherical Mercator tile calculations.
- [geopandas](https://geopandas.org/): For geographic data manipulation.
- [shapely](https://shapely.readthedocs.io/): For geometric operations.
- [fiona](https://fiona.readthedocs.io/): For file format support.

In [2]:
!pip install mercantile geopandas shapely fiona

  Using cached mercantile-1.2.1-py3-none-any.whl.metadata (4.8 kB)
Using cached mercantile-1.2.1-py3-none-any.whl (14 kB)


## 2. Declare Variables

Input parameters for the grid generation process. Define the zoom level, output format, AOI file path, and the output name.

In [5]:
# The zoom level for the tile grid
zoom = 17

# Output format: 'geojson' or 'csv'
output_format = 'geojson'

# Path to the Area of Interest (AOI) GeoJSON file
aoi_path = '../data/CZU_Fire_Perimeter_-2212360052269988636.geojson'

# Name for the output file
output_name = 'Z18_CZU_tile_boundary_grid'

## 3. Helper Functions and Processing Logic

Load the AOI, ensure it is in the correct CRS (WGS84), and find all intersecting tiles at the specified zoom level.

In [6]:
import mercantile
import geopandas as gpd
from shapely.geometry import shape, box, mapping
import pandas as pd

# Load the AOI
aoi_gdf = gpd.read_file(aoi_path)

# Ensure AOI is in WGS84 (EPSG:4326)
if aoi_gdf.crs != 'EPSG:4326':
    aoi_gdf = aoi_gdf.to_crs('EPSG:4326')

# Combine all features in the AOI for bounds calculation
bounds = aoi_gdf.total_bounds # [west, south, east, north]

# Get all tiles within the bounding box of the AOI
tiles = list(mercantile.tiles(*bounds, zoom))

print(f"Generated {len(tiles)} tiles at zoom level {zoom}")

Generated 11115 tiles at zoom level 17


## 4. Constructing the Tile Data

For each tile, extract the bounding box, geometry, and calculate metadata such as `zxy_val`, `url_suffix`, and the centroid. Then, create the final GeoDataFrame.

In [7]:
tile_data = []

for tile in tiles:
    # Get the bounding box of the tile
    t_bounds = mercantile.bounds(tile) # [west, south, east, north]
    t_geom = box(t_bounds.west, t_bounds.south, t_bounds.east, t_bounds.north)
    
    # Calculate properties
    zxy_val = f"{tile.z},{tile.x},{tile.y}"
    url_suffix = f"{tile.z}/{tile.x}/{tile.y}"
    
    # Upper left (west, north) and Lower right (east, south)
    upper_left = f"{t_bounds.west},{t_bounds.north}"
    lower_right = f"{t_bounds.east},{t_bounds.south}"
    
    # Centroid
    centroid = t_geom.centroid
    centroid_val = f"{centroid.x},{centroid.y}"
    
    tile_data.append({
        'zxy_val': zxy_val,
        'z_val': int(tile.z),
        'x_val': int(tile.x),
        'y_val': int(tile.y),
        'url_suffix': url_suffix,
        'upper_left': upper_left,
        'lower_right': lower_right,
        'centroid': centroid_val,
        'geometry': t_geom
    })

# Create a GeoDataFrame
grid_gdf = gpd.GeoDataFrame(tile_data, crs='EPSG:4326')

## 5. Exporting Results

Depending on the `output_format` variable, save the boundaries to GeoJSON or CSV. For CSV, convert the geometry to WKT (Well-Known Text) for compatibility.

In [8]:
output_path = f"../output/{output_name}.{output_format if output_format == 'geojson' else 'csv'}"

if output_format == 'geojson':
    grid_gdf.to_file(output_path, driver='GeoJSON')
    print(f"Exported to GeoJSON: {output_path}")
elif output_format == 'csv':
    # Convert geometry to WKT for CSV/BigQuery ingest
    grid_gdf['geometry'] = grid_gdf['geometry'].apply(lambda x: x.wkt)
    grid_gdf.to_csv(output_path, index=False)
    print(f"Exported to CSV with WKT: {output_path}")

Exported to GeoJSON: ../output/Z18_CZU_tile_boundary_grid.geojson
